# Memory-Split PoC — Facts in Context (no DB retrieval)

**Question.** At a fixed parameter budget, does a model that *offloads facts to context* — never memorizing them — reason better than a dense twin that stores them in weights?

One corpus, two arms, **one toggle**: every fact is in a `Context:` block; only the loss mask on the fact **value** differs. DENSE (loss on) memorizes; SPLIT (masked) offloads to context. Three open-book tasks: fact-QA, reason-over-facts (yes/no comparison), and pure reasoning (`puremath`; the operation's *definition* is the context).

**Four eval conditions** (both arms × closed-book/+context): DENSE @ closed-book · DENSE + context · SPLIT @ closed-book · SPLIT + context.

**Scaling note.** Capacity ~ params, so a *small* model is needed for a fixed token budget to actually stress capacity (a 162M model sits at ~3% capacity utilization overnight; toy/mini is where crowding can appear). But too small and it can't reason at all. So **run the pilot first** (§4) to pick the smallest size that still reasons above chance, then set `MODEL` for the full run.

GPU runtime; paste a TrueFoundry token (only used to grade fact-QA). Checkpoints persist to Drive.

## 1. Get the code (syncs to the latest branch state)

In [ ]:
import os
REPO_URL = "https://github.com/sidvenkatayogi/Memory-Split.git"
BRANCH = "poc/optimal-retriever"
if os.path.basename(os.getcwd()) != "Memory-Split":
    if not os.path.isdir("Memory-Split"):
        !git clone --branch {BRANCH} --single-branch {REPO_URL}
    %cd Memory-Split
!git fetch -q origin {BRANCH} && git reset --hard -q FETCH_HEAD
!git log --oneline -1

In [ ]:
!pip install -q tiktoken openai
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['POC_PERSIST_DIR'] = '/content/drive/MyDrive/memory_split_poc'
os.makedirs(os.environ['POC_PERSIST_DIR'], exist_ok=True)
print('persist dir ->', os.environ['POC_PERSIST_DIR'])

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("TrueFoundry token: ")
os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
os.environ["POC_GPT_MODEL"] = "openai-group/gpt-5.6-sol"
import sys; sys.path.insert(0, ".")
from evals.gpt_oracle import GatewayClient
print("gateway smoke ->", GatewayClient().smoke())

## 4. Pilot — find the smallest model that still reasons
Trains reasoning-only (no fact dose) and checks if the model beats chance. Run for a couple of sizes (`micro` 6.7M · `mini` 13.7M · `toy` 29M) and pick the **smallest that's ABOVE floor** — that's the one where a fixed dose stresses capacity most while reasoning is still measurable.

In [ ]:
!python scripts/poc_run.py --stage pilot --model mini --device auto --steps 1500
!python scripts/poc_run.py --stage pilot --model toy  --device auto --steps 1500

In [ ]:
# Set the size for the full run to the smallest pilot that cleared the floor:
MODEL = "toy"   # e.g. 'mini' if it was usable, else 'toy'
print('full run will use model =', MODEL)

## 5. Build the corpus (with the fact dose) — dial `--max-facts` to set the load

In [ ]:
!python scripts/poc_run.py --stage build

## 6. Train the matched twins, then evaluate + report
Same corpus/model/init/budget; only the value loss-mask differs. Reuses checkpoints on Drive (`--fresh` to retrain).

In [ ]:
!python scripts/poc_run.py --stage train  --device auto --model {MODEL} --steps 4000 --ckpt-minutes 5
!python scripts/poc_run.py --stage eval   --device auto --model {MODEL} --judge
!python scripts/poc_run.py --stage report --model {MODEL}

In [ ]:
import json, os
from IPython.display import Image, display
persist = os.environ.get('POC_PERSIST_DIR', 'data/poc')
print(json.dumps(json.load(open(f'{persist}/poc_results.json')), indent=2))
display(Image(f'{persist}/poc_figure.png'))